# Training RuBERT for Russian Sentiment Analysis

This notebook trains a RuBERT model on the Russian Sentiment Dataset.
The trained model can be downloaded and used locally for XAI.

## 1. Setup and Install Dependencies
Install required packages

In [1]:
%%capture

!pip install transformers torch scikit-learn pandas numpy matplotlib tqdm pyyaml -q

## 2. Import Libraries

In [2]:
import os
import gc
import random
import yaml
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoConfig, get_linear_schedule_with_warmup
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Define Model Architecture (RuBERTClassifier)

In [3]:
class RuBERTClassifier(nn.Module):
    """
    RuBERT model fine-tuned for 3-class sentiment classification.
    """
    
    def __init__(
        self,
        model_name: str = "DeepPavlov/rubert-base-cased",
        num_labels: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.num_labels = num_labels
        
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_attentions=output_attentions,
        )
        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        
        if output_attentions:
            attentions = outputs.attentions
            stacked = torch.stack(attentions)
            avg_attn = stacked.mean(dim=0).mean(dim=1)
            cls_attn = avg_attn[:, 0, :]
            return logits, cls_attn
        
        return logits

## 4. Data Preprocessing

In [4]:
def load_and_split_data(
    csv_path: str,
    text_column: str = "text",
    label_column: str = "label",
    val_split: float = 0.2,
    random_state: int = 42,
):
    """Load CSV and split into train/val."""
    df = pd.read_csv(csv_path)
    
    print(f"Total samples: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Clean labels
    df[label_column] = pd.to_numeric(df[label_column], errors="coerce")
    df = df.dropna(subset=[label_column])
    df[label_column] = df[label_column].astype(int)
    
    # Remove duplicates
    original_len = len(df)
    df = df.drop_duplicates(subset=[text_column], keep="first")
    if len(df) < original_len:
        print(f"Removed {original_len - len(df)} duplicate texts")
    
    # Split
    train_df, val_df = train_test_split(
        df,
        test_size=val_split,
        random_state=random_state,
        stratify=df[label_column],
    )
    
    print(f"\nTrain size: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"Val size: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
    
    print("\nTrain class distribution:")
    print(train_df[label_column].value_counts().sort_index().to_dict())
    print("Val class distribution:")
    print(val_df[label_column].value_counts().sort_index().to_dict())
    
    return train_df, val_df


def prepare_texts_labels(df, text_column="text", label_column="label"):
    """Extract texts and labels from DataFrame."""
    texts = df[text_column].astype(str).str.strip().tolist()
    labels = df[label_column].values
    
    # Filter empty texts
    valid_indices = [i for i, t in enumerate(texts) if t]
    texts = [texts[i] for i in valid_indices]
    labels = labels[valid_indices]
    
    print(f"Final dataset size: {len(texts)} samples")
    return texts, labels


class SentimentDataset(Dataset):
    """PyTorch Dataset for sentiment classification."""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

## 5. Evaluation Function

In [5]:
def evaluate(model, data_loader, device):
    """Evaluate model accuracy."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return correct / total if total > 0 else 0.0

## 6. Set Random Seed for Reproducibility

In [6]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
print(f"Random seed set to {SEED}")

Random seed set to 42


## 7. Load Dataset

**Important:** Make sure to add your `sentiment_dataset.csv` to the notebook:
- Click "Add Data" → "Upload" → select your CSV file
- Or use the Kaggle dataset: `mar1mba/russian-sentiment-dataset`

In [7]:
CSV_PATH = "/kaggle/input/datasets/mar1mba/russian-sentiment-dataset/sentiment_dataset.csv"

# Load and split
train_df, val_df = load_and_split_data(
    CSV_PATH,
    text_column="text",
    label_column="label",
    val_split=0.2,
    random_state=SEED,
)

# Prepare texts and labels
train_texts, train_labels = prepare_texts_labels(train_df)
val_texts, val_labels = prepare_texts_labels(val_df)

Total samples: 290458
Columns: ['text', 'label', 'src']
Removed 68 duplicate texts

Train size: 232312 (80.0%)
Val size: 58078 (20.0%)

Train class distribution:
{0: 77250, 1: 77492, 2: 77570}
Val class distribution:
{0: 19312, 1: 19373, 2: 19393}
Final dataset size: 232312 samples
Final dataset size: 58077 samples


## 8. Configuration

In [8]:
config = {
    "model": {
        "name": "DeepPavlov/rubert-base-cased",
        "num_labels": 3,
        "max_length": 128,
        "batch_size": 16,  # Can be increased on GPU
        "epochs": 5,
        "learning_rate": 2e-5,
    },
    "data": {
        "label_names": ["neutral", "positive", "negative"],
    }
}

print("Configuration:")
print(yaml.dump(config, allow_unicode=True))

Configuration:
data:
  label_names:
  - neutral
  - positive
  - negative
model:
  batch_size: 16
  epochs: 5
  learning_rate: 2.0e-05
  max_length: 128
  name: DeepPavlov/rubert-base-cased
  num_labels: 3



## 9. Create Data Loaders

In [9]:
tokenizer = AutoTokenizer.from_pretrained(config["model"]["name"])

train_dataset = SentimentDataset(
    train_texts, train_labels, tokenizer,
    max_length=config["model"]["max_length"]
)

val_dataset = SentimentDataset(
    val_texts, val_labels, tokenizer,
    max_length=config["model"]["max_length"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["model"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["model"]["batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Train batches: 14520
Val batches: 3630


## 10. Initialize Model, Optimizer, and Scheduler

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = RuBERTClassifier(
    model_name=config["model"]["name"],
    num_labels=config["model"]["num_labels"],
)
model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["model"]["learning_rate"],
)

total_steps = len(train_loader) * config["model"]["epochs"]
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

criterion = torch.nn.CrossEntropyLoss()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Using device: cuda


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters: 177,855,747
Trainable parameters: 177,855,747


## 11. Training Loop

In [12]:
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

best_accuracy = 0.0

for epoch in range(config["model"]["epochs"]):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['model']['epochs']}")
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")
    
    # Validation
    accuracy = evaluate(model, val_loader, device)
    print(f"  Validation Accuracy: {accuracy:.4f}")
    
    # Save best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_path = checkpoint_dir / "rubert_sentiment_best.pt"
        torch.save(model.state_dict(), best_model_path)
        print(f"  Best model saved to {best_model_path} (acc={accuracy:.4f})")
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save final model
final_model_path = checkpoint_dir / "rubert_sentiment.pt"
torch.save(model.state_dict(), final_model_path)
print(f"\nFinal model saved to {final_model_path}")

# Save training info
training_info = {
    "config": config,
    "best_accuracy": float(best_accuracy),
    "seed": SEED,
    "device": device,
}
with open(checkpoint_dir / "training_config.yaml", "w", encoding="utf-8") as f:
    yaml.dump(training_info, f, allow_unicode=True, default_flow_style=False)

print(f"\nBest validation accuracy: {best_accuracy:.4f}")

Epoch 1/5: 100%|██████████| 14520/14520 [1:36:55<00:00,  2.50it/s, loss=0.4426]

Epoch 1 - Average Loss: 0.6602


  Validation Accuracy: 0.7242
  Best model saved to checkpoints/rubert_sentiment_best.pt (acc=0.7242)


Epoch 2/5: 100%|██████████| 14520/14520 [1:36:58<00:00,  2.50it/s, loss=0.2497]

Epoch 2 - Average Loss: 0.5342


...

Epoch 4 - Average Loss: 0.3149


  Validation Accuracy: 0.7437


Epoch 5/5: 100%|██████████| 14520/14520 [1:37:01<00:00,  2.49it/s, loss=0.3059]

Epoch 5 - Average Loss: 0.2326


  Validation Accuracy: 0.7391

Final model saved to checkpoints/rubert_sentiment.pt

Best validation accuracy: 0.7482


## 12. Quick Test

In [13]:
# Test the model on a few examples
model.eval()
test_texts = [
    "Фильм был отличный, очень понравилась игра актёров!",
    "Ужасный сервис, больше никогда не приду сюда.",
    "Обычный продукт, ничего особенного не заметил.",
]

label_names = config["data"]["label_names"]

for text in test_texts:
    encoding = tokenizer(
        text,
        max_length=config["model"]["max_length"],
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    
    print(f"\nText: {text}")
    print(f"Prediction: {label_names[pred]} ({probs[0][pred].item():.2%})")
    print(f"Probabilities: {dict(zip(label_names, probs[0].cpu().numpy()))}")


Text: Фильм был отличный, очень понравилась игра актёров!
Prediction: neutral (96.78%)
Probabilities: {'neutral': np.float32(0.9678467), 'positive': np.float32(0.018194264), 'negative': np.float32(0.0139590725)}

Text: Ужасный сервис, больше никогда не приду сюда.
Prediction: negative (99.92%)
Probabilities: {'neutral': np.float32(0.0006987243), 'positive': np.float32(7.191408e-05), 'negative': np.float32(0.9992293)}

Text: Обычный продукт, ничего особенного не заметил.
Prediction: neutral (89.44%)
Probabilities: {'neutral': np.float32(0.89439124), 'positive': np.float32(0.027171234), 'negative': np.float32(0.07843759)}


## 13. Download Trained Model

In [14]:
# Create a zip file with all checkpoints
import zipfile

zip_path = "rubert_sentiment_checkpoints.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file_path in checkpoint_dir.glob("*"):
        zipf.write(file_path, arcname=file_path.name)

print(f"Created {zip_path} with:")
for file_path in checkpoint_dir.glob("*"):
    print(f"  - {file_path.name}")

# Download the zip file
from IPython.display import FileLink
FileLink(zip_path)

Created rubert_sentiment_checkpoints.zip with:
  - training_config.yaml
  - rubert_sentiment.pt
  - rubert_sentiment_best.pt


/kaggle/working/rubert_sentiment_checkpoints.zip